In [79]:
import os
#os.chdir("../")
%pwd

'/home/abood/Desktop/ML_Model_Monitoring_System_for_Credit_Risk'

In [80]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/aboodcs/ML_Model_Monitoring_System_for_Credit_Risk.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "aboodcs"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "6101bda6b23aa9ef440664cb2f0aac4cb03e1bcf"


In [81]:
from pathlib import Path
from dataclasses import dataclass

In [82]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_path: Path
    mlflow_uri: str
    target_column: str
    all_params: dict 

In [83]:
from src.creditrisk.constants import *
from src.creditrisk.utils.common import read_yaml, create_directories,save_json

In [84]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet 
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            metric_file_path = config.metric_file_path,
            all_params = params,
            mlflow_uri = "https://dagshub.com/aboodcs/ML_Model_Monitoring_System_for_Credit_Risk.mlflow",
            target_column = schema.name)
        return model_evaluation_config

In [85]:
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
from urllib.parse import urlparse

In [86]:

class ModelEvaluation:
    def __init__(self,config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual , pred):
        rmse = np.sqrt(mean_squared_error(actual , pred))
        mae = mean_absolute_error(actual , pred)
        r2 = r2_score(actual , pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_x = pd.read_csv(self.config.test_data_path)
        test_y = pd.read_csv(str(self.config.test_data_path).replace("X_test.csv", "y_test.csv"))
        test_y = test_y.values.ravel()
        model = joblib.load(self.config.model_path)

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

            #saving metrices as local
            scores = {
                "rmse": rmse,
                "mae": mae,
                "r2": r2}
            save_json(path=Path(self.config.metric_file_path), data=scores)

            mlflow.log_params(self.config.all_params)
            
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, name="model", registered_model_name="ElasticNetWineModel")
            else:
                mlflow.sklearn.log_model(model, name="model")

In [87]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-05-02 23:13:40,835: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-02 23:13:40,838: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-02 23:13:40,842: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-02 23:13:40,844: INFO: common]: created directory at: artifacts
[2026-05-02 23:13:40,845: INFO: common]: created directory at: artifacts/model_evaluation
[2026-05-02 23:13:42,282: INFO: common]: json file saved at: artifacts/model_evaluation/metrics.json


2026/05/02 23:13:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticNetWineModel' already exists. Creating a new version of this model...
2026/05/02 23:14:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetWineModel, version 4
Created version '4' of model 'ElasticNetWineModel'.


🏃 View run traveling-shark-573 at: https://dagshub.com/aboodcs/ML_Model_Monitoring_System_for_Credit_Risk.mlflow/#/experiments/0/runs/8c3fbb47626c4ef0802f827bacee61d9
🧪 View experiment at: https://dagshub.com/aboodcs/ML_Model_Monitoring_System_for_Credit_Risk.mlflow/#/experiments/0
